In [90]:
import sys
import asyncio

if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

print("Event loop policy siap.")

Event loop policy siap.


In [91]:
!pip install pandas openpyxl beautifulsoup4 playwright
!playwright install chromium


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [92]:
import re
import time
import random
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

In [93]:
BASE_URL = "https://sinta.kemdiktisaintek.go.id"

INPUT_FILE = r"D:\Proyek FEB\source data\unique_author_2_sumber_sinta.xlsx"
INPUT_SHEET = "Unique Authors"
AUTHOR_COLUMN = "Normalized Key"

OUTPUT_FILE = "garuda_sinta_nasional_2020_2026.xlsx"

START_YEAR = 2020
END_YEAR = 2026

# main system

In [ ]:
script_code = r'''
import re
import time
import random
import argparse
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError


BASE_URL = "https://sinta.kemdiktisaintek.go.id"

INPUT_FILE = r"D:\Proyek FEB\source data\unique_author_2_sumber_sinta.xlsx"
INPUT_SHEET = "Unique Authors"
AUTHOR_COLUMN = "Normalized Key"

OUTPUT_FILE = "garuda_sinta_nasional_2020_2026.xlsx"

START_YEAR = 2020
END_YEAR = 2026

USER_DATA_DIR = "sinta_browser_profile"

AFFILIATION_KEYWORDS = [
    "universitas airlangga",
    "univ airlangga",
    "airlangga",
    "airlngga",
    "unair",
]

DEPARTMENT_KEYWORDS = [
    # Manajemen
    "ilmu manajemen",
    "sains manajemen",
    "manajemen",

    # Ekonomi Islam
    "ilmu ekonomi islam",
    "sains ekonomi islam",
    "ekonomi islam",
    "ekonomi syariah",
    "ekonomi syari ah",
    "ekonomi syari'ah",

    # Akuntansi / Akuntan
    "ilmu akuntansi",
    "sains akuntansi",
    "akuntansi",
    "akuntan",
    "pendidikan profesi akuntan",
    "pendidikan profesi akuntan profesi",

    # Ekonomi
    "ilmu ekonomi",
    "sains ekonomi",
    "ekonomi pembangunan",

    # Tambahan FEB
    "pengembangan sumber daya manusia",
]

UNKNOWN_DEPARTMENT_VALUES = [
    "",
    "-",
    "unknown",
    "none",
    "null",
    "n/a",
    "na",
]

FINAL_COLUMNS = [
    "Title",
    "Authors",
    "Journal",
    "SINTA",
    "Tahun",
    "Volume / Issue",
    "LINK DOI/ARTIKEL",
]


def clean_text(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def normalize_text(text):
    return clean_text(text).lower()


def is_unknown_department(text):
    text_lower = normalize_text(text)

    if text_lower in UNKNOWN_DEPARTMENT_VALUES:
        return True

    if "unknown" in text_lower:
        return True

    return False


def extract_sinta_id_from_text(text):
    text = clean_text(text)
    match = re.search(r"SINTA\s*ID\s*:\s*(\d+)", text, re.I)
    return match.group(1) if match else ""


def extract_year(text):
    text = clean_text(text)
    match = re.search(r"\b(2020|2021|2022|2023|2024|2025|2026)\b", text)
    return int(match.group(1)) if match else None


def extract_sinta_rank(text):
    text = clean_text(text)
    match = re.search(r"Accred\s*:\s*(Sinta\s*\d+)", text, re.I)
    return clean_text(match.group(1)).title() if match else ""


def extract_doi_link(text):
    text = clean_text(text)

    match = re.search(r"DOI\s*:\s*([^\s]+)", text, re.I)
    if match:
        doi = match.group(1).strip().rstrip(".,;)")
        if doi.lower().startswith("http"):
            return doi
        return f"https://doi.org/{doi}"

    match = re.search(r"\b(10\.\d{4,9}/[^\s]+)", text, re.I)
    if match:
        doi = match.group(1).strip().rstrip(".,;)")
        return f"https://doi.org/{doi}"

    return ""


def extract_volume_issue(text):
    text = clean_text(text)

    patterns = [
        r"(Vol\.?\s*\d+[A-Za-z]?\s*,?\s*No\.?\s*\d+[A-Za-z]?)",
        r"(Vol\.?\s*\d+[A-Za-z]?\s*,?\s*Number\s*\d+[A-Za-z]?)",
        r"(Volume\s*\d+[A-Za-z]?\s*,?\s*Number\s*\d+[A-Za-z]?)",
        r"(Volume\s*\d+[A-Za-z]?\s*,?\s*No\.?\s*\d+[A-Za-z]?)",
        r"(Vol\.?\s*\d+[A-Za-z]?\s*/\s*No\.?\s*\d+[A-Za-z]?)",
        r"(Vol\.?\s*\d+[A-Za-z]?)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.I)
        if match:
            return clean_text(match.group(1))

    return ""


def remove_volume_from_journal(journal_text):
    journal_text = clean_text(journal_text)

    split_patterns = [
        r"\s+Vol\.?\s*\d+",
        r"\s+Volume\s*\d+",
    ]

    for pattern in split_patterns:
        parts = re.split(pattern, journal_text, flags=re.I)
        if parts and parts[0]:
            return clean_text(parts[0])

    return journal_text


def read_authors(limit=None):
    df = pd.read_excel(INPUT_FILE, sheet_name=INPUT_SHEET)

    if AUTHOR_COLUMN not in df.columns:
        raise ValueError(
            f"Kolom '{AUTHOR_COLUMN}' tidak ditemukan. "
            f"Kolom tersedia: {list(df.columns)}"
        )

    authors = (
        df[AUTHOR_COLUMN]
        .dropna()
        .astype(str)
        .map(clean_text)
        .drop_duplicates()
        .tolist()
    )

    authors = [a for a in authors if a]

    if limit:
        authors = authors[:limit]

    return authors


def launch_context(playwright, headless=False):
    context = playwright.chromium.launch_persistent_context(
        USER_DATA_DIR,
        headless=headless,
        slow_mo=80,
        viewport={"width": 1366, "height": 768},
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        args=[
            "--disable-blink-features=AutomationControlled",
            "--disable-infobars",
            "--start-maximized",
            "--no-sandbox",
            "--disable-dev-shm-usage",
        ],
    )

    if context.pages:
        page = context.pages[0]
    else:
        page = context.new_page()

    return context, page


def manual_login(minutes=5):
    with sync_playwright() as p:
        context, page = launch_context(p, headless=False)

        page.goto(BASE_URL, wait_until="domcontentloaded", timeout=60000)

        print("=" * 80)
        print("LOGIN MANUAL SINTA")
        print("1. Browser akan terbuka.")
        print("2. Silakan login ke akun SINTA.")
        print("3. Setelah login berhasil, biarkan browser tetap terbuka.")
        print(f"4. Script akan menunggu {minutes} menit lalu menyimpan session.")
        print("=" * 80)

        time.sleep(minutes * 60)

        context.close()

    print("Session login tersimpan di folder:", USER_DATA_DIR)


def is_affiliation_match(text):
    text_lower = normalize_text(text)
    return any(keyword in text_lower for keyword in AFFILIATION_KEYWORDS)


def is_department_match(text):
    text_lower = normalize_text(text)
    return any(keyword in text_lower for keyword in DEPARTMENT_KEYWORDS)


def search_sinta_id(page, author_name):
    search_url = f"{BASE_URL}/authors?q={quote_plus(author_name)}"

    page.goto(search_url, wait_until="domcontentloaded", timeout=60000)
    page.wait_for_timeout(random.randint(1500, 3000))

    try:
        html = page.content()
        body_text = clean_text(page.inner_text("body"))
    except Exception:
        return ""

    if "Data Not Found" in body_text:
        print(f"SKIP: Data Not Found -> {author_name}")
        return ""

    soup = BeautifulSoup(html, "html.parser")

    candidate_blocks = soup.select(".list-item")

    if not candidate_blocks:
        candidate_blocks = soup.select(".profile-side, .row, .card")

    candidates = []

    for block in candidate_blocks:
        block_text = clean_text(block.get_text(" ", strip=True))

        sinta_match = re.search(r"SINTA\s*ID\s*:\s*(\d+)", block_text, re.I)
        if not sinta_match:
            continue

        sinta_id = sinta_match.group(1)

        name_tag = block.select_one(".profile-name")
        affil_tag = block.select_one(".profile-affil")
        dept_tag = block.select_one(".profile-dept")

        candidate_name = clean_text(name_tag.get_text(" ", strip=True)) if name_tag else ""
        candidate_affil = clean_text(affil_tag.get_text(" ", strip=True)) if affil_tag else ""
        candidate_dept = clean_text(dept_tag.get_text(" ", strip=True)) if dept_tag else ""

        affil_check_text = candidate_affil if candidate_affil else block_text
        dept_check_text = candidate_dept

        affil_ok = is_affiliation_match(affil_check_text)
        dept_ok = is_department_match(dept_check_text)
        dept_unknown = is_unknown_department(candidate_dept)

        candidates.append({
            "sinta_id": sinta_id,
            "name": candidate_name,
            "affil": candidate_affil,
            "dept": candidate_dept,
            "affil_ok": affil_ok,
            "dept_ok": dept_ok,
            "dept_unknown": dept_unknown,
            "block_text": block_text,
        })

    if not candidates:
        print(f"SKIP: SINTA ID tidak terbaca -> {author_name}")
        return ""

    unair_candidates = [c for c in candidates if c["affil_ok"]]

    if not unair_candidates:
        print(f"SKIP: Ada hasil search, tapi tidak ada afiliasi Universitas Airlangga -> {author_name}")

        for c in candidates:
            print("  Lewati kandidat non-UNAIR:")
            print(f"  Nama    : {c['name'] if c['name'] else '-'}")
            print(f"  Afiliasi: {c['affil'] if c['affil'] else '-'}")
            print(f"  Prodi   : {c['dept'] if c['dept'] else '-'}")
            print(f"  SINTA ID: {c['sinta_id']}")

        return ""

    # Prioritas utama: kandidat UNAIR + prodi FEB
    dept_matched = [c for c in unair_candidates if c["dept_ok"]]

    if dept_matched:
        c = dept_matched[0]

        print("  Kandidat cocok UNAIR + Prodi FEB:")
        print(f"  Nama    : {c['name'] if c['name'] else '-'}")
        print(f"  Afiliasi: {c['affil'] if c['affil'] else '-'}")
        print(f"  Prodi   : {c['dept'] if c['dept'] else '-'}")
        print(f"  SINTA ID: {c['sinta_id']}")

        return c["sinta_id"]

    # Kalau hanya satu kandidat UNAIR dan prodinya kosong/unknown, tetap ambil
    # Kalau prodinya muncul tapi bukan FEB, skip.
    if len(unair_candidates) == 1:
        c = unair_candidates[0]

        if c["dept_unknown"]:
            print("  Kandidat UNAIR tunggal dengan prodi unknown/kosong diambil:")
            print(f"  Nama    : {c['name'] if c['name'] else '-'}")
            print(f"  Afiliasi: {c['affil'] if c['affil'] else '-'}")
            print(f"  Prodi   : {c['dept'] if c['dept'] else 'UNKNOWN / kosong'}")
            print(f"  SINTA ID: {c['sinta_id']}")

            return c["sinta_id"]

    print(f"SKIP: Kandidat UNAIR ada, tapi prodi tidak cocok FEB -> {author_name}")

    for c in unair_candidates:
        print("  Lewati kandidat UNAIR non-FEB:")
        print(f"  Nama    : {c['name'] if c['name'] else '-'}")
        print(f"  Afiliasi: {c['affil'] if c['affil'] else '-'}")
        print(f"  Prodi   : {c['dept'] if c['dept'] else 'UNKNOWN / kosong'}")
        print(f"  SINTA ID: {c['sinta_id']}")

    return ""


def parse_garuda_page(html, author_name):
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    article_items = soup.select(".ar-list-item")

    for item in article_items:
        title_tag = item.select_one(".ar-title a")
        if not title_tag:
            continue

        title = clean_text(title_tag.get_text(" ", strip=True))
        article_link = title_tag.get("href", "")

        if article_link.startswith("/"):
            article_link = BASE_URL + article_link

        full_text = clean_text(item.get_text(" ", strip=True))

        year = extract_year(full_text)
        if year is None:
            continue

        if year < START_YEAR or year > END_YEAR:
            continue

        sinta_rank = extract_sinta_rank(full_text)

        doi_link = extract_doi_link(full_text)
        link_final = doi_link if doi_link else article_link

        pub_tag = item.select_one(".ar-pub")

        journal_raw = ""
        if pub_tag:
            journal_raw = clean_text(pub_tag.get_text(" ", strip=True))
        else:
            for a in item.select("a"):
                a_text = clean_text(a.get_text(" ", strip=True))
                if re.search(r"\bVol\.?\s*\d+|\bVolume\s*\d+", a_text, re.I):
                    journal_raw = a_text
                    break

        if not journal_raw:
            journal_raw = full_text

        volume_issue = extract_volume_issue(journal_raw)
        journal = remove_volume_from_journal(journal_raw)

        rows.append({
            "Title": title,
            "Authors": author_name,
            "Journal": journal,
            "SINTA": sinta_rank,
            "Tahun": year,
            "Volume / Issue": volume_issue,
            "LINK DOI/ARTIKEL": link_final,
        })

    return rows


def go_to_next_page(page):
    selectors = [
        "nav[aria-label='pagination-sample'] li.page-item:not(.disabled) a:has-text('Next')",
        "ul.pagination li.page-item:not(.disabled) a:has-text('Next')",
        "li.page-item:not(.disabled) a:has-text('Next')",
    ]

    for selector in selectors:
        try:
            next_btn = page.locator(selector)
            if next_btn.count() > 0 and next_btn.first.is_visible():
                old_text = clean_text(page.inner_text("body"))

                next_btn.first.click(timeout=5000)
                page.wait_for_timeout(random.randint(2500, 4500))

                new_text = clean_text(page.inner_text("body"))

                if old_text != new_text:
                    return True
        except Exception:
            pass

    return False


def scrape_garuda_by_sinta_id(page, sinta_id, author_name):
    garuda_url = f"{BASE_URL}/authors/profile/{sinta_id}/?view=garuda"

    page.goto(garuda_url, wait_until="domcontentloaded", timeout=60000)
    page.wait_for_timeout(random.randint(2500, 4500))

    all_rows = []
    seen_page_signatures = set()

    for page_no in range(1, 50):
        print(f"  Ambil Garuda page {page_no}")

        page.wait_for_timeout(random.randint(1000, 2000))

        html = page.content()
        signature = clean_text(page.inner_text("body"))[:2000]

        if signature in seen_page_signatures:
            print("  Stop pagination: halaman berulang.")
            break

        seen_page_signatures.add(signature)

        rows = parse_garuda_page(html, author_name)
        all_rows.extend(rows)

        has_next = go_to_next_page(page)

        if not has_next:
            print("  Stop pagination: tidak ada tombol Next aktif.")
            break

    return all_rows


def save_output(all_rows):
    df_out = pd.DataFrame(all_rows, columns=FINAL_COLUMNS)

    if not df_out.empty:
        df_out = df_out.drop_duplicates(
            subset=[
                "Title",
                "Authors",
                "Journal",
                "Tahun",
                "LINK DOI/ARTIKEL",
            ],
            keep="first"
        )

        df_out = df_out.sort_values(
            by=["Authors", "Tahun", "Title"],
            ascending=[True, False, True]
        )

    df_out.to_excel(OUTPUT_FILE, index=False)

    print("=" * 80)
    print("Selesai.")
    print("Total baris output:", len(df_out))
    print("File tersimpan:", OUTPUT_FILE)

    return df_out


def scrape(limit=None, headless=False):
    authors = read_authors(limit=limit)

    print("Total author diproses:", len(authors))

    all_rows = []

    with sync_playwright() as p:
        context, page = launch_context(p, headless=headless)

        for idx, author_name in enumerate(authors, start=1):
            print("=" * 80)
            print(f"[{idx}/{len(authors)}] Search author: {author_name}")

            try:
                sinta_id = search_sinta_id(page, author_name)

                if not sinta_id:
                    continue

                rows = scrape_garuda_by_sinta_id(
                    page=page,
                    sinta_id=sinta_id,
                    author_name=author_name
                )

                if not rows:
                    print(f"Tidak ada artikel Garuda 2020-2026: {author_name}")
                    continue

                all_rows.extend(rows)
                print(f"Artikel diambil: {len(rows)}")

                page.wait_for_timeout(random.randint(2000, 5000))

            except PlaywrightTimeoutError:
                print(f"TIMEOUT: {author_name}")
                continue

            except Exception as e:
                print(f"ERROR pada author {author_name}: {e}")
                continue

        context.close()

    save_output(all_rows)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--login",
        action="store_true",
        help="Buka browser untuk login manual SINTA dan simpan session."
    )

    parser.add_argument(
        "--login-minutes",
        type=int,
        default=5,
        help="Durasi tunggu login manual dalam menit."
    )

    parser.add_argument(
        "--limit",
        type=int,
        default=None,
        help="Batasi jumlah author untuk testing."
    )

    parser.add_argument(
        "--headless",
        action="store_true",
        help="Jalankan browser tanpa tampilan."
    )

    args = parser.parse_args()

    if args.login:
        manual_login(minutes=args.login_minutes)
    else:
        scrape(limit=args.limit, headless=args.headless)
'''

with open("scrape_sinta_garuda_login.py", "w", encoding="utf-8") as f:
    f.write(script_code)

print("File script berhasil dibuat: scrape_sinta_garuda_login.py")

File script berhasil dibuat: scrape_sinta_garuda_login.py


In [95]:
from pathlib import Path
import re

path = Path("scrape_sinta_garuda_login.py")
code = path.read_text(encoding="utf-8")

old = r'''
def launch_context(playwright, headless=False):
    context = playwright.chromium.launch_persistent_context(
        USER_DATA_DIR,
        channel="chrome",
        headless=headless,
        slow_mo=120,
        viewport={"width": 1366, "height": 768},
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        args=[
            "--disable-blink-features=AutomationControlled",
            "--disable-infobars",
            "--start-maximized",
        ]
    )

    if context.pages:
        page = context.pages[0]
    else:
        page = context.new_page()

    return context, page
'''

new = r'''
def launch_context(playwright, headless=False):
    context = playwright.chromium.launch_persistent_context(
        USER_DATA_DIR,
        headless=headless,
        slow_mo=120,
        viewport={"width": 1366, "height": 768},
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        args=[
            "--disable-blink-features=AutomationControlled",
            "--disable-infobars",
            "--start-maximized",
            "--no-sandbox",
            "--disable-dev-shm-usage",
        ]
    )

    if context.pages:
        page = context.pages[0]
    else:
        page = context.new_page()

    return context, page
'''

if old not in code:
    print("Blok lama tidak ditemukan. Kita pakai regex fallback.")
    code = re.sub(
        r"def launch_context\(playwright, headless=False\):.*?def manual_login",
        new + "\n\n\ndef manual_login",
        code,
        flags=re.S
    )
else:
    code = code.replace(old, new)

path.write_text(code, encoding="utf-8")

print("Patch selesai: script sekarang memakai Chromium bawaan Playwright.")

Blok lama tidak ditemukan. Kita pakai regex fallback.
Patch selesai: script sekarang memakai Chromium bawaan Playwright.


In [96]:
# import os

# os.system("taskkill /F /IM chrome.exe")
# os.system("taskkill /F /IM chromium.exe")
# os.system("taskkill /F /IM msedge.exe")

In [97]:
# import shutil
# from pathlib import Path

# profile_dir = Path("sinta_browser_profile")

# if profile_dir.exists():
#     shutil.rmtree(profile_dir)
#     print("Session lama dihapus.")
# else:
#     print("Session lama tidak ada.")

In [98]:
# !python scrape_sinta_garuda_login.py --login --login-minutes 5

In [99]:
# !python scrape_sinta_garuda_login.py 

# author miss

In [100]:
import pandas as pd
import re
from pathlib import Path

RAW_FILE = r"D:\Proyek FEB\source data\unique_author_2_sumber_sinta.xlsx"
RAW_SHEET = "Unique Authors"
RAW_AUTHOR_COL = "Normalized Key"

SCRAPED_FILE = r"D:\Proyek FEB\sinta\garuda_sinta_nasional_2020_2026.xlsx"
SCRAPED_AUTHOR_COL = "Authors"

MISS_INPUT_FILE = r"D:\Proyek FEB\source data\miss_author_for_scrape.xlsx"


def norm_author(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"[^a-z\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


df_raw = pd.read_excel(RAW_FILE, sheet_name=RAW_SHEET)
df_scraped = pd.read_excel(SCRAPED_FILE)

df_raw["author_norm"] = df_raw[RAW_AUTHOR_COL].apply(norm_author)
df_scraped["author_norm"] = df_scraped[SCRAPED_AUTHOR_COL].apply(norm_author)

raw_unique = (
    df_raw[[RAW_AUTHOR_COL, "author_norm"]]
    .dropna()
    .drop_duplicates(subset=["author_norm"])
)

scraped_unique = (
    df_scraped[[SCRAPED_AUTHOR_COL, "author_norm"]]
    .dropna()
    .drop_duplicates(subset=["author_norm"])
)

df_miss = raw_unique[
    ~raw_unique["author_norm"].isin(scraped_unique["author_norm"])
].copy()

df_miss_output = df_miss[[RAW_AUTHOR_COL]].copy()

with pd.ExcelWriter(MISS_INPUT_FILE, engine="openpyxl") as writer:
    df_miss_output.to_excel(writer, sheet_name="Unique Authors", index=False)

print("Total raw author:", len(raw_unique))
print("Total author sudah terscrape:", len(scraped_unique))
print("Total author miss:", len(df_miss_output))
print("File miss:", MISS_INPUT_FILE)

df_miss_output.head(50)

Total raw author: 171
Total author sudah terscrape: 127
Total author miss: 44
File miss: D:\Proyek FEB\source data\miss_author_for_scrape.xlsx


,Normalized Key
2,amalia rizki
6,basuki
10,devi sulistyo kalanjati
17,hamidah
20,ika atma kurniawanti
25,khusnul prasetyo
26,la ode sabaruddin
28,moh nasih
33,okta sindhu hartadinata
39,sri ningsih


In [101]:
from pathlib import Path
import re

script_path = Path("scrape_sinta_garuda_login.py")
code = script_path.read_text(encoding="utf-8")

# Ubah input ke file miss
code = re.sub(
    r'INPUT_FILE\s*=\s*r?".*?"',
    lambda m: r'INPUT_FILE = r"D:\Proyek FEB\source data\miss_author_for_scrape.xlsx"',
    code
)

# Output sementara khusus author miss
code = re.sub(
    r'OUTPUT_FILE\s*=\s*".*?"',
    lambda m: 'OUTPUT_FILE = "garuda_sinta_nasional_2020_2026_MISS_ONLY.xlsx"',
    code
)

new_search_block = r'''
def extract_candidates_from_search_page(html):
    soup = BeautifulSoup(html, "html.parser")

    candidate_blocks = soup.select(".list-item")

    if not candidate_blocks:
        candidate_blocks = soup.select(".profile-side, .row, .card")

    candidates = []

    for block in candidate_blocks:
        block_text = clean_text(block.get_text(" ", strip=True))

        sinta_match = re.search(r"SINTA\s*ID\s*:\s*(\d+)", block_text, re.I)
        if not sinta_match:
            continue

        sinta_id = sinta_match.group(1)

        name_tag = block.select_one(".profile-name")
        affil_tag = block.select_one(".profile-affil")
        dept_tag = block.select_one(".profile-dept")

        candidate_name = clean_text(name_tag.get_text(" ", strip=True)) if name_tag else ""
        candidate_affil = clean_text(affil_tag.get_text(" ", strip=True)) if affil_tag else ""
        candidate_dept = clean_text(dept_tag.get_text(" ", strip=True)) if dept_tag else ""

        affil_check_text = candidate_affil if candidate_affil else block_text
        dept_check_text = candidate_dept

        affil_ok = is_affiliation_match(affil_check_text)
        dept_ok = is_department_match(dept_check_text)

        try:
            dept_unknown = is_unknown_department(candidate_dept)
        except NameError:
            dept_lower = clean_text(candidate_dept).lower()
            dept_unknown = dept_lower in ["", "-", "unknown", "none", "null", "n/a", "na"]

        candidates.append({
            "sinta_id": sinta_id,
            "name": candidate_name,
            "affil": candidate_affil,
            "dept": candidate_dept,
            "affil_ok": affil_ok,
            "dept_ok": dept_ok,
            "dept_unknown": dept_unknown,
            "block_text": block_text,
        })

    return candidates


def go_to_next_search_page(page):
    selectors = [
        "nav[aria-label='pagination-sample'] li.page-item:not(.disabled) a:has-text('Next')",
        "ul.pagination li.page-item:not(.disabled) a:has-text('Next')",
        "li.page-item:not(.disabled) a:has-text('Next')",
    ]

    for selector in selectors:
        try:
            next_btn = page.locator(selector)

            if next_btn.count() > 0 and next_btn.first.is_visible():
                old_text = clean_text(page.inner_text("body"))

                next_btn.first.click(timeout=5000)
                page.wait_for_timeout(random.randint(2000, 4000))

                new_text = clean_text(page.inner_text("body"))

                if old_text != new_text:
                    return True

        except Exception:
            pass

    return False


def choose_best_sinta_candidate(candidates, author_name):
    if not candidates:
        print(f"SKIP: SINTA ID tidak terbaca -> {author_name}")
        return ""

    unair_candidates = [c for c in candidates if c["affil_ok"]]

    if not unair_candidates:
        print(f"SKIP: Ada hasil search, tapi tidak ada afiliasi Universitas Airlangga -> {author_name}")
        return ""

    # Prioritas 1: UNAIR + prodi FEB
    dept_matched = [c for c in unair_candidates if c["dept_ok"]]

    if dept_matched:
        c = dept_matched[0]

        print("  Kandidat cocok UNAIR + Prodi FEB:")
        print(f"  Nama    : {c['name'] if c['name'] else '-'}")
        print(f"  Afiliasi: {c['affil'] if c['affil'] else '-'}")
        print(f"  Prodi   : {c['dept'] if c['dept'] else '-'}")
        print(f"  SINTA ID: {c['sinta_id']}")

        return c["sinta_id"]

    # Prioritas 2: hanya satu kandidat UNAIR dan prodi kosong/unknown
    if len(unair_candidates) == 1:
        c = unair_candidates[0]

        if c["dept_unknown"]:
            print("  Kandidat UNAIR tunggal dengan prodi unknown/kosong diambil:")
            print(f"  Nama    : {c['name'] if c['name'] else '-'}")
            print(f"  Afiliasi: {c['affil'] if c['affil'] else '-'}")
            print(f"  Prodi   : {c['dept'] if c['dept'] else 'UNKNOWN / kosong'}")
            print(f"  SINTA ID: {c['sinta_id']}")

            return c["sinta_id"]

    print(f"SKIP: Kandidat UNAIR ada, tapi prodi tidak cocok FEB -> {author_name}")

    for c in unair_candidates:
        print("  Lewati kandidat UNAIR non-FEB:")
        print(f"  Nama    : {c['name'] if c['name'] else '-'}")
        print(f"  Afiliasi: {c['affil'] if c['affil'] else '-'}")
        print(f"  Prodi   : {c['dept'] if c['dept'] else 'UNKNOWN / kosong'}")
        print(f"  SINTA ID: {c['sinta_id']}")

    return ""


def search_sinta_id(page, author_name):
    search_url = f"{BASE_URL}/authors?q={quote_plus(author_name)}"

    page.goto(search_url, wait_until="domcontentloaded", timeout=60000)
    page.wait_for_timeout(random.randint(1500, 3000))

    try:
        body_text = clean_text(page.inner_text("body"))
    except Exception:
        return ""

    if "Data Not Found" in body_text:
        print(f"SKIP: Data Not Found -> {author_name}")
        return ""

    all_candidates = []
    seen_page_signatures = set()

    for search_page_no in range(1, 30):
        print(f"  Cek hasil search author page {search_page_no}")

        page.wait_for_timeout(random.randint(1000, 2000))

        html = page.content()
        body_text = clean_text(page.inner_text("body"))
        signature = body_text[:2000]

        if signature in seen_page_signatures:
            print("  Stop search pagination: halaman berulang.")
            break

        seen_page_signatures.add(signature)

        page_candidates = extract_candidates_from_search_page(html)

        if page_candidates:
            print(f"  Kandidat di page ini: {len(page_candidates)}")
            all_candidates.extend(page_candidates)
        else:
            print("  Tidak ada kandidat terbaca di page ini.")

        has_next = go_to_next_search_page(page)

        if not has_next:
            print("  Stop search pagination: tidak ada tombol Next aktif.")
            break

    return choose_best_sinta_candidate(all_candidates, author_name)
'''

# PENTING:
# Pakai lambda supaya backslash dalam new_search_block tidak dianggap escape oleh re.sub
code = re.sub(
    r"def search_sinta_id\(page, author_name\):.*?def parse_garuda_page",
    lambda m: new_search_block + "\n\n\ndef parse_garuda_page",
    code,
    flags=re.S
)

script_path.write_text(code, encoding="utf-8")

print("Script sudah dipatch: input miss + search author pagination + output MISS_ONLY.")

Script sudah dipatch: input miss + search author pagination + output MISS_ONLY.


In [102]:
# !python scrape_sinta_garuda_login.py --login --login-minutes 5

In [103]:
!python scrape_sinta_garuda_login.py 

Total author diproses: 44
[1/44] Search author: amalia rizki
  Cek hasil search author page 1
  Kandidat di page ini: 4
  Stop search pagination: tidak ada tombol Next aktif.
  Kandidat cocok UNAIR + Prodi FEB:
  Nama    : zB_Qwu4AAAAJ AMALIA RIZKI
  Afiliasi: Universitas Airlangga
  Prodi   : Akuntansi (S1)
  SINTA ID: 6659386
  Ambil Garuda page 1
  Stop pagination: tidak ada tombol Next aktif.
Tidak ada artikel Garuda 2020-2026: amalia rizki
[2/44] Search author: basuki
  Cek hasil search author page 1
  Kandidat di page ini: 10
  Cek hasil search author page 2
  Kandidat di page ini: 10
  Cek hasil search author page 3
  Kandidat di page ini: 10
  Cek hasil search author page 4
  Kandidat di page ini: 10
  Cek hasil search author page 5
  Kandidat di page ini: 10
  Cek hasil search author page 6
  Kandidat di page ini: 10
  Cek hasil search author page 7
  Kandidat di page ini: 10
  Cek hasil search author page 8
  Kandidat di page ini: 10
  Cek hasil search author page 9
  Kandida

In [104]:
import pandas as pd
from pathlib import Path
import shutil

# File hasil scraping lama / berhasil sebelumnya
OLD_FILE = r"D:\Proyek FEB\sinta\garuda_sinta_nasional_2020_2026.xlsx"
# File hasil scraping khusus miss
MISS_FILE = r"D:\Proyek FEB\sinta\garuda_sinta_nasional_2020_2026_MISS_ONLY.xlsx"

# Output final gabungan
FINAL_FILE = "garuda_sinta_nasional_2020_2026_fixed.xlsx"

# Backup dulu hasil lama
BACKUP_FILE = "backup_garuda_sinta_nasional_2020_2026.xlsx"

if Path(OLD_FILE).exists():
    shutil.copy(OLD_FILE, BACKUP_FILE)
    print("Backup dibuat:", BACKUP_FILE)
else:
    raise FileNotFoundError(f"File lama tidak ditemukan: {OLD_FILE}")

if not Path(MISS_FILE).exists():
    raise FileNotFoundError(f"File hasil miss tidak ditemukan: {MISS_FILE}")


final_columns = [
    "Title",
    "Authors",
    "Journal",
    "SINTA",
    "Tahun",
    "Volume / Issue",
    "LINK DOI/ARTIKEL",
]

df_old = pd.read_excel(OLD_FILE)
df_miss = pd.read_excel(MISS_FILE)

# Pastikan hanya kolom final yang dipakai
df_old = df_old[final_columns].copy()
df_miss = df_miss[final_columns].copy()

# Gabungkan
df_final = pd.concat(
    [df_old, df_miss],
    ignore_index=True
)

# Bersihkan data kosong
df_final["Title"] = df_final["Title"].astype(str).str.strip()
df_final["Authors"] = df_final["Authors"].astype(str).str.strip()
df_final["Journal"] = df_final["Journal"].astype(str).str.strip()
df_final["LINK DOI/ARTIKEL"] = df_final["LINK DOI/ARTIKEL"].astype(str).str.strip()

# Drop duplicate
df_final = df_final.drop_duplicates(
    subset=[
        "Title",
        "Authors",
        "Journal",
        "Tahun",
        "LINK DOI/ARTIKEL",
    ],
    keep="first"
)

# Urutkan
df_final = df_final.sort_values(
    by=["Authors", "Tahun", "Title"],
    ascending=[True, False, True]
)

# Simpan final
df_final.to_excel(FINAL_FILE, index=False)

print("Data lama:", len(df_old))
print("Data miss baru:", len(df_miss))
print("Data final gabungan:", len(df_final))
print("File final:", FINAL_FILE)

df_final.head(20)

Backup dibuat: backup_garuda_sinta_nasional_2020_2026.xlsx
Data lama: 2303
Data miss baru: 517
Data final gabungan: 2820
File final: garuda_sinta_nasional_2020_2026_fixed.xlsx


,Title,Authors,Journal,SINTA,Tahun,Volume / Issue,LINK DOI/ARTIKEL
0,SURVIVAL ANALYSIS TO DETERMINE AGE TO GIVE FIR...,achmad sjafii,Jurnal Biometrika dan Kependudukan (Journal of...,Sinta 2,2021,Vol. 10 No. 2,https://doi.org/10.20473/jbk.v10i2.2021.144-152
1,Pertumbuhan Tanpa Pemerataan: Analisis Fenomen...,achmad solihin,Paradoks : Jurnal Ilmu Ekonomi,Sinta 4,2026,Vol. 9 No. 2,https://doi.org/10.57178/paradoks.v9i2.2254
2,The Effect of Fiscal Policy and Macroeconomic ...,achmad solihin,"Journal of Mathematics Instruction, Social Res...",Sinta 2,2026,Vol. 5 No. 1,https://doi.org/10.58421/misro.v5i1.1297
3,PENGARUH DANA BANTUAN OPERASIONAL KESEHATAN DA...,achmad solihin,"Jurnal Ilmiah Manajemen, Ekonomi, & Akuntansi ...",Sinta 4,2025,Vol 9 No 1,https://doi.org/10.31955/mea.v9i1.5145
4,Segmentasi Pelanggan Berdasarkan Model LRFM Me...,achmad solihin,Jurnal Informatika: Jurnal Pengembangan IT,Sinta 3,2025,"Vol 10, No 3",https://doi.org/10.30591/jpit.v10i3.8735
5,Strategi BUMDes Kaduara Timur dalam mengembang...,achmad solihin,Jurnal Inovasi Hasil Pengabdian Masyarakat (JI...,Sinta 3,2025,Vol 8 No 2,https://doi.org/10.33474/jipemas.v8i2.22636
6,The Effect of Infrastructure and Economic Grow...,achmad solihin,International Journal of Economics Development...,Sinta 3,2025,Vol. 6 No. 3,https://doi.org/10.37385/ijedr.v5i3.7583
7,DAMPAK PROGRAM BANTUAN TUNAI TERHADAP PERSEPSI...,achmad solihin,"Jurnal Ilmiah Manajemen, Ekonomi, & Akuntansi ...",Sinta 4,2024,Vol 8 No 2,https://doi.org/10.31955/mea.v8i2.4155
8,Determinants of Spending Efficiency for Educat...,achmad solihin,JEJAK,Sinta 2,2024,Vol. 17 No. 1,https://doi.org/10.15294/jejak.v17i1.5620
9,EFISIENSI BELANJA PEMERINTAH DAERAH TERHADAP K...,achmad solihin,"Jurnal Ilmiah Manajemen, Ekonomi, & Akuntansi ...",Sinta 4,2024,Vol 8 No 3,https://doi.org/10.31955/mea.v8i3.4493
